# The Jupyter plugin

AML-GraphMiner plugs into notebooks in two ways:

* its results **display themselves**: a scan shows its alerts per motif, an alert its
  evidence, an explanation its verdict and an interactive graph (hover the arrows and the
  circles), a benchmark its table of metrics;
* the extension `%load_ext aml_graphminer` adds three **magics**: `%amlgm` runs any command of
  the command line inside the kernel, `%aml_scan` scans a DataFrame of the notebook and
  `%aml_explain` explains a transaction.

Everything below runs on synthetic data in about a minute, without any download. The files it
writes go to `data/` and `runs/` next to the notebook.

In [1]:
%pip install -q "aml-graphminer[viz]"

## 1. The command line inside the notebook

`%amlgm` takes exactly the commands and options of the `amlgm` command line. It runs in the
interpreter of the kernel, so there is no PATH to set up and the version of the package is the
one installed in the kernel. Here it writes a synthetic IBM-like dataset and scans it.

In [2]:
%load_ext aml_graphminer
%amlgm generate data --accounts 600 --background 5000 --seed 5

wrote data\Synth-Small_Trans.csv and data\Synth-Small_Patterns.txt
5099 transactions, 50 laundering


In [3]:
%amlgm scan data/Synth-Small_Trans.csv --top 5

63 alert(s) on 5099 transactions
 alert_id pattern  score  n_tx  n_accounts               start                 end  total_amount
       59   cycle  0.883     6           6 2022-09-03 12:22:00 2022-09-04 07:30:00     38722.430
       56   cycle  0.838     3           3 2022-09-06 10:55:00 2022-09-08 16:13:00       189.887
       58   cycle  0.836     3           3 2022-09-05 11:21:00 2022-09-08 10:14:00       382.605
       53   cycle  0.836     3           3 2022-09-03 14:27:00 2022-09-06 18:35:00       563.940
       54   cycle  0.823     3           3 2022-09-03 22:44:00 2022-09-08 08:34:00       318.810

Transaction-level metrics:
{
  "threshold": 0.0,
  "n": 5099.0,
  "n_pos": 50.0,
  "pr_auc": 0.2792,
  "roc_auc": 0.8222,
  "f1": 0.1623,
  "precision": 0.0921,
  "recall": 0.68,
  "recall_at_k": 0.12,
  "recall_at_100": 0.52,
  "recall_at_1000": 0.68,
  "recall_at_0.1pct": 0.1,
  "recall_at_1pct": 0.12
}

Per-detector precision:
       pattern  n_alerts  n_tx_covered  tx_precision

## 2. Results that display themselves

`%aml_scan` scans a DataFrame of the notebook (or a file) and returns the `ScanResult`, which is
shown as a summary per motif and a table of the strongest alerts. The same display appears for
`aml.PatternScanner().scan(df)` left at the end of a cell: no extension is needed for it.

In [4]:
import aml_graphminer as aml
from aml_graphminer.detectors.bundle import MotifDetector
from aml_graphminer.ingestion import load_dataset

df = load_dataset("data/Synth-Small_Trans.csv")
print(f"{len(df):,} transactions, {int(df['label'].sum())} of them laundering")

5,099 transactions, 50 of them laundering


In [5]:
%aml_scan df

#,pattern,score,transactions,accounts,amount,period,accounts involved
1,cycle,0.883,6,6,"38,722.43",2022-09-03 12:22 → 2022-09-04 07:30,"3_000004BF, 2_000025D2, 12_000025E6, +3"
2,cycle,0.838,3,3,189.89,2022-09-06 10:55 → 2022-09-08 16:13,"8_00001312, 3_000004BF, 1_000012AA"
3,cycle,0.836,3,3,382.61,2022-09-05 11:21 → 2022-09-08 10:14,"10_00000E80, 4_00000388, 12_0000182D"
4,cycle,0.836,3,3,563.94,2022-09-03 14:27 → 2022-09-06 18:35,"8_00001312, 4_00000388, 1_000012AA"
5,cycle,0.823,3,3,318.81,2022-09-03 22:44 → 2022-09-08 08:34,"8_00001312, 6_00000D61, 7_000010D7"
6,cycle,0.821,3,3,524.14,2022-09-03 10:30 → 2022-09-08 14:08,"4_00000388, 10_00000E80, 12_0000182D"
7,cycle,0.816,4,4,393.76,2022-09-04 19:13 → 2022-09-08 02:38,"8_00001312, 10_00000E80, 4_00000388, +1"
8,smurfing,0.809,8,9,"5,305.92",2022-09-06 10:49 → 2022-09-07 10:10,"5_00001D18, 4_00001C48, 5_00001E4D, +6"
9,cycle,0.806,3,3,"1,116.96",2022-09-02 15:18 → 2022-09-07 12:53,"8_00001312, 12_0000182D, 9_00001AC0"
10,cycle,0.800,3,3,536.87,2022-09-01 18:25 → 2022-09-06 18:35,"8_00001312, 4_00000388, 1_000012AA"


Every alert carries its evidence — the facts a compliance officer writes in a suspicious
activity report:

In [6]:
result = aml.PatternScanner().scan(df)
max(result.alerts, key=result.weight)

,
score,0.883
period,2022-09-03 12:22 → 2022-09-04 07:30
total amount,"38,722.43"
transactions,"6: 1390, 1411, 1562, 1689, 1731, 1793"
accounts,"6: 3_000004BF, 2_000025D2, 12_000025E6, 2_000025FC, 8_0000260B, 12_00002612"
length,6
span_seconds,"68,880.00"
first_amount,"38,722.43"
last_amount,"35,798.59"
amount_retained,0.9245


## 3. Explaining a transaction

`%aml_explain MODEL DATA TX` takes a detector of the notebook or a model folder written by
`amlgm train`. First the motif detector, calibrated on the validation period: its explanation
is exact (the alerts that contain the transaction). In the graph the explained transaction is
red; hover an arrow for the transaction, a circle for the account.

In [7]:
motifs = MotifDetector.fit(df, fractions=(0.6, 0.2, 0.2))
tx = int(motifs.score(df).idxmax())
print("transaction with the highest score:", tx)

transaction with the highest score: 3462


In [8]:
%aml_explain motifs df {tx}

tx,sender → receiver,amount,time,importance
3472,8_00001312 → 6_00001BA6,"7,734.52",2022-09-07 09:23:00,+0.647
3497,8_00001312 → 9_00002645,"7,285.69",2022-09-07 09:52:00,+0.647
3524,6_00001BA6 → 4_00002620,"7,504.70",2022-09-07 10:25:00,+0.647
3572,9_00002645 → 4_00002620,"7,130.20",2022-09-07 11:29:00,+0.647
3581,4_00001A5D → 4_00002620,"7,269.44",2022-09-07 11:45:00,+0.647


Now a detector trained by the command line and read back from its folder, on the strongest
**cycle** of the scan. The ring of accounts is plain in the graph, yet the verdict can be *no
alert*: the calibration measured on the validation period how often each motif was really
laundering, and cycles were rarely so in these data. An explanation always gives both the
evidence and the verdict.

In [9]:
%amlgm train data/Synth-Small_Trans.csv --model patterns --out runs/motifs
cycle = max((a for a in result.alerts if a.pattern == "cycle"), key=result.weight)
cycle_tx = cycle.tx_ids[0]
print(f"cycle of {len(cycle.accounts)} accounts, raw alert score {cycle.score:.3f}")
print(
    f"precision of cycle alerts on the validation period: {motifs.scanner.calibration['cycle']:.2f}"
)

saved patterns detector to runs/motifs
{
  "model_type": "patterns",
  "detectors": [
    "community",
    "cycle",
    "fan_in",
    "fan_out",
    "gather_scatter",
    "scatter_gather",
    "smurfing"
  ],
  "threshold": 0.646594774222274,
  "calibrated": true,
  "metrics": {
    "threshold": 0.646594774222274,
    "n": 1020.0,
    "n_pos": 6.0,
    "pr_auc": 0.0058823529411764705,
    "roc_auc": 0.46844181459566075,
    "f1": 0.0,
    "precision": 0.0,
    "recall": 0.0,
    "recall_at_k": 0.0,
    "recall_at_100": 0.0,
    "recall_at_1000": 0.6666666666666666,
    "recall_at_0.1pct": 0.0,
    "recall_at_1pct": 0.0
  }
}
cycle of 6 accounts, raw alert score 0.883
precision of cycle alerts on the validation period: 0.14


In [10]:
%aml_explain runs/motifs df {cycle_tx}

tx,sender → receiver,amount,time,importance
1411,2_000025D2 → 12_000025E6,"37,910.49",2022-09-03 12:50:00,+0.126
1562,12_000025E6 → 2_000025FC,"37,496.98",2022-09-03 16:30:00,+0.126
1689,2_000025FC → 8_0000260B,"36,530.58",2022-09-03 21:18:00,+0.126
1731,8_0000260B → 12_00002612,"36,261.63",2022-09-04 02:17:00,+0.126
1793,12_00002612 → 3_000004BF,"35,798.59",2022-09-04 07:30:00,+0.126


## 4. Benchmarks

A `BenchmarkReport` shows the mean (and, with several seeds, the standard deviation) of the
test metrics of every model. On a sample this small (a handful of laundering transactions in
the test period) the numbers only illustrate the display: notebook 02 runs the real benchmark.

In [11]:
from aml_graphminer.evaluation.benchmark import run_benchmark

run_benchmark(df, models=("patterns", "hgb"), seeds=(0,))

BenchmarkReport(2 models, 2 runs)

## 5. The same views outside a notebook

The HTML comes from `aml_graphminer.jupyter`: it can be saved, for instance to attach an
explanation to a case file.

In [12]:
from pathlib import Path

from aml_graphminer.explainers import explain_transaction
from aml_graphminer.jupyter import explanation_html

page = Path("runs/why.html")
page.write_text(explanation_html(explain_transaction(motifs, df, tx)), encoding="utf-8")
print(f"{page} written ({page.stat().st_size:,} bytes)")

runs\why.html written (7,470 bytes)
